In [2]:
import os
import h5py
import cv2
from ultralytics import YOLO
import yaml
import os

def convert_svhn_to_yolo(mat_path, img_dir, split):
    images_out = f'dataset/images/{split}'
    labels_out = f'dataset/labels/{split}'
    
    os.makedirs(images_out, exist_ok=True)
    os.makedirs(labels_out, exist_ok=True)
    
    f = h5py.File(mat_path, 'r')
    bboxes = f['/digitStruct/bbox']
    names = f['/digitStruct/name']
    
    for i in range(len(names)):
        name_ref = names[i][0]
        img_name = ''.join([chr(c[0]) for c in f[name_ref][()]])
        
        src_path = os.path.join(img_dir, img_name)
        img = cv2.imread(src_path)
        if img is None:
            continue
        
        H, W = img.shape[:2]
        bbox_ref = bboxes[i][0]
        
        lefts, tops, widths, heights = [], [], [], []
        
        for key, arr in zip(['left','top','width','height'], [lefts, tops, widths, heights]):
            attr = f[bbox_ref][key]
            if len(attr) > 1:
                arr.extend([f[attr[j][0]][0][0] for j in range(len(attr))])
            else:
                arr.append(attr[0][0])
        
        x_min = max(0, min(lefts))
        y_min = max(0, min(tops))
        x_max = min(W, max([l+w for l,w in zip(lefts,widths)]))
        y_max = min(H, max([t+h for t,h in zip(tops,heights)]))
        
        x_center = (x_min + x_max) / (2.0 * W)
        y_center = (y_min + y_max) / (2.0 * H)
        w = (x_max - x_min) / W
        h = (y_max - y_min) / H
        
        dst_img_path = os.path.join(images_out, img_name)
        cv2.imwrite(dst_img_path, img)
        
        txt_name = img_name.replace('.png', '.txt')
        with open(os.path.join(labels_out, txt_name), 'w') as f_out:
            f_out.write(f"0 {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n")
    f.close()

convert_svhn_to_yolo('train/digitStruct.mat', 'train', 'train')
convert_svhn_to_yolo('test/digitStruct.mat', 'test', 'val')

print("Train:", len(os.listdir('dataset/images/train')))
print("Val:", len(os.listdir('dataset/images/val')))

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\pirog\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Train: 33402
Val: 13068


In [3]:
current_dir = os.path.abspath('.')

data = {
    'path': os.path.abspath('dataset'),
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,
    'names': ['number']
}

with open('svhn_data.yaml', 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

In [5]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = YOLO('yolov8n.pt')

results = model.train(
    data='svhn_data.yaml',
    epochs=20,
    imgsz=640,
    batch=16,
    name='numbers_model',
    device=device
)

engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=svhn_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=numbers_model, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, project=None, rect=False, res

In [6]:
best_model = YOLO('runs/detect/numbers_model/weights/best.pt')
metrics = best_model.val()

print(f"mAP@50: {metrics.box.map50:.3f}")
print(f"mAP@50-95: {metrics.box.map:.3f}")

Ultralytics 8.4.31  Python-3.12.5 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 187.7320.1 MB/s, size: 34.8 KB)
val: Scanning C:\Users\pirog\OneDrive\Desktop\машинки\cv5\dataset\labels\val.cache... 13068 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 13068/13068  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 817/817 10.6it/s 1:170.1sss
                   all      13068      13068      0.913      0.887      0.934       0.55
Speed: 0.8ms preprocess, 1.6ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to C:\Users\pirog\OneDrive\Desktop\\cv5\runs\detect\val
mAP@50: 0.934
mAP@50-95: 0.550


In [11]:
predictions = best_model.predict(
    source='my_photos',
    save=True,
    imgsz=1024,
    conf=0.25,
    augment=True,
    iou=0.4
)


image 1/10 C:\Users\pirog\OneDrive\Desktop\\cv5\my_photos\photo_2026-03-30_15-10-39 (2).jpg: 768x1024 1 number, 113.1ms
image 2/10 C:\Users\pirog\OneDrive\Desktop\\cv5\my_photos\photo_2026-03-30_15-10-39.jpg: 768x1024 1 number, 92.7ms
image 3/10 C:\Users\pirog\OneDrive\Desktop\\cv5\my_photos\photo_2026-03-30_15-10-40 (2).jpg: 768x1024 1 number, 77.1ms
image 4/10 C:\Users\pirog\OneDrive\Desktop\\cv5\my_photos\photo_2026-03-30_15-10-40.jpg: 768x1024 (no detections), 54.2ms
image 5/10 C:\Users\pirog\OneDrive\Desktop\\cv5\my_photos\photo_2026-03-30_15-10-41 (2).jpg: 768x1024 2 numbers, 50.2ms
image 6/10 C:\Users\pirog\OneDrive\Desktop\\cv5\my_photos\photo_2026-03-30_15-10-41.jpg: 768x1024 1 number, 49.8ms
image 7/10 C:\Users\pirog\OneDrive\Desktop\\cv5\my_photos\photo_2026-03-30_15-10-42 (2).jpg: 1024x768 1 number, 49.7ms
image 8/10 C:\Users\pirog\OneDrive\Desktop\\cv5\my_photos\photo_2026-03-30_15-10-42.jpg: 1024x768 2 numbers, 49.2ms
image 9/10 C:\Users\pirog\OneDrive\Desktop\\cv5\my_ph

### В гитхаб загружены не все фотографии т.к. на некоторых присутствуют чужие лица/чужие номерные знаки машины. Всего фотографий было сделано 10, если необходимо, то отправлю лично.